# Support Vector Machines

## Introduction

This notebook continues the modeling workflow for predicting customer churn by introducing a Support Vector Machine (SVM) as a margin-based alternative to the previously explored models.

In earlier stages, logistic regression provided an interpretable linear baseline, while tree-based methods (decision trees, random forests, and gradient boosting) introduced non-linear modeling through recursive partitioning of the feature space. These approaches highlighted different ways of capturing structure in the data, either through global linear relationships or localized decision rules.

Support Vector Machines offer a complementary perspective by focusing on the concept of decision boundaries and margins. Rather than modeling probabilities directly or partitioning the feature space, an SVM identifies a boundary that separates classes while maximizing the distance between them. This margin-based formulation often leads to strong generalization performance, particularly in high-dimensional settings.

In this notebook, the analysis focuses on two controlled variants of the SVM: a linear model and a non-linear model using the radial basis function (RBF) kernel. The linear SVM provides a direct comparison to logistic regression under a different optimization objective, while the RBF kernel introduces a flexible, non-linear decision boundary without requiring explicit feature engineering. This targeted approach allows us to evaluate the impact of margin-based learning and controlled non-linearity without introducing unnecessary model complexity.

Unlike tree-based methods, SVMs are sensitive to the scale of input features. As a result, feature standardization becomes an essential component of the modeling pipeline to ensure that all variables contribute appropriately to the optimization process.

From an interpretability standpoint, SVMs differ from both linear models and decision trees. The model does not yield easily interpretable coefficients or explicit decision rules. Instead, predictions are determined by a subset of training observations known as support vectors, which define the position and orientation of the decision boundary. While this limits direct interpretability, it provides a geometric understanding of how the model separates classes.

The objective of this stage is to assess whether a margin-based approach can achieve competitive or improved performance relative to both linear and tree-based models. In particular, we evaluate the extent to which controlled non-linearity, introduced through the RBF kernel, contributes to predictive performance.

To ensure consistency with previous experiments, the same evaluation framework is retained. Model training and hyperparameter tuning are conducted using cross-validation within the training data, while a separate validation set is used for model selection. A final hold-out test set remains reserved for unbiased performance evaluation.

As in prior notebooks, the focus extends beyond classification accuracy to the quality of predicted probabilities and the model’s ability to rank customers by churn risk. Metrics such as ROC-AUC, PR-AUC, and threshold-based performance measures are used consistently across all models.

The results of this notebook will provide a margin-based benchmark and further enrich the comparison between linear, tree-based, and kernel-based modeling approaches.

## Table of Contents

1. [Modeling Considerations](#modeling-considerations)
2. [Linear SVM](#linear-svm)
   - [Initial Grid](#initial-grid-definition-and-cross-validation-linear)
3. [RBF Kernel SVM](#rbf-kernel-svm)
   - [Initial Grid](#initial-grid-definition-and-cross-validation-rbf)
4. [Test Set Evaluation](#test-set-evaluation)
5. [Executive Summary](#executive-summary)

## Modeling Considerations

Before fitting the Support Vector Machine (SVM) model, we outline several key considerations that influence model behavior and evaluation.

---

### Class Imbalance

The target variable exhibits moderate class imbalance, with approximately 26.5% of customers churning.

SVMs are sensitive to class imbalance because the optimization objective focuses on maximizing the margin between classes. In imbalanced settings, the decision boundary may shift toward the minority class, prioritizing correct classification of the majority class while underrepresenting churners.

To address this, accuracy is not used as a primary evaluation metric. Instead, we focus on recall, precision, F1 score, and PR-AUC, which better capture performance on the minority class. In addition, class weights can be incorporated into the model to penalize misclassification of the minority class more heavily, helping to balance the decision boundary.

---

### Feature Scaling

Feature scaling is essential for SVM models.

Unlike tree-based methods, SVMs rely on distance-based calculations when determining the optimal decision boundary. Features with larger numerical ranges can dominate the optimization process, leading to suboptimal margins and degraded performance.

To mitigate this, all numerical features are standardized prior to model training. This ensures that each feature contributes proportionally to the model and allows the optimization procedure to converge more reliably.

---

### Feature Representation

SVMs operate on a fixed feature space and do not inherently perform feature selection or transformation.

Categorical variables are encoded using one-hot encoding, and numerical features are retained in their continuous form. Unlike decision trees, SVMs do not learn feature interactions or non-linear transformations explicitly. Instead, any non-linear structure is captured through the use of kernel functions.

In this notebook, feature representation remains consistent with earlier models, ensuring comparability while allowing the SVM to leverage both linear and kernel-based decision boundaries.

---

### Model Complexity and Regularization

SVMs control model complexity through the regularization parameter $ C $ and, in the case of non-linear kernels, the kernel-specific parameters such as $ \gamma $ for the RBF kernel.

The parameter $ C $ governs the trade-off between maximizing the margin and minimizing classification errors. Smaller values of $ C $ produce wider margins with stronger regularization, while larger values allow the model to fit the training data more closely, increasing the risk of overfitting.

For the RBF kernel, the parameter $ \gamma $ controls the flexibility of the decision boundary. High values lead to highly localized, complex boundaries, while low values produce smoother, more global decision surfaces.

Careful tuning of these parameters is essential to balance model flexibility and generalization performance.

---

### Probability Estimation

SVMs do not inherently produce probability estimates. Instead, they generate decision scores based on the distance from the decision boundary.

To obtain calibrated probabilities, an additional step is required. In this notebook, probability estimates are derived using built-in calibration methods, enabling consistent evaluation using metrics such as ROC-AUC and PR-AUC, as well as threshold-based analysis.

---

### Validation Strategy and Iterative Model Refinement

Hyperparameter tuning and model refinement are performed iteratively, which introduces a risk of overfitting to evaluation data if not properly controlled. While cross-validation provides robust estimates of model performance within the training data, repeated use of the same data for model comparison and refinement can lead to optimistic bias.

To address this, the training data is further split into a dedicated training subset and a separate validation set. Cross-validation is applied within the training subset to evaluate candidate hyperparameter configurations, while the validation set is used to guide model refinement and select the final model.

This separation ensures that model selection decisions are not based on the same data used for model fitting, reducing the risk of selection bias. A final hold-out test set is reserved exclusively for unbiased performance evaluation and is not used at any stage of model tuning.

---

### Summary

Support Vector Machines introduce a margin-based approach to classification, offering a fundamentally different perspective compared to both linear models and tree-based methods. While they can achieve strong generalization performance, they require careful handling of feature scaling, class imbalance, and hyperparameter tuning.

The modeling approach therefore emphasizes proper preprocessing, controlled model complexity, and rigorous validation. By focusing on both linear and RBF variants, the analysis isolates the impact of margin-based learning and controlled non-linearity while maintaining a clear and interpretable modeling framework.

With these considerations in place, we proceed to defining and training the SVM model.

## Model Definition and Hyperparameter Tuning

The Support Vector Machine model is implemented using the `SVC` class from scikit-learn. Unlike logistic regression, which estimates coefficients directly, or tree-based methods, which learn hierarchical decision rules, SVMs define classification boundaries through an optimization problem that seeks to maximize the margin between classes.

To maintain a structured and interpretable modeling workflow, the analysis is conducted in two stages. We begin with a linear SVM to establish a margin-based baseline, and then extend the model to a non-linear specification using the radial basis function (RBF) kernel. This staged approach allows us to isolate the impact of margin-based learning from the additional flexibility introduced by non-linear decision boundaries.

In the first stage, the linear SVM is defined using a single key hyperparameter, $C$, which controls the trade-off between maximizing the margin and minimizing classification errors. Smaller values of $C$ enforce stronger regularization and wider margins, while larger values allow the model to fit the training data more closely.

In the second stage, the RBF kernel is introduced to capture non-linear relationships. In addition to $C$, the model includes the parameter $\gamma$, which determines the influence of individual observations on the decision boundary. Lower values of $\gamma$ result in smoother, more global decision surfaces, while higher values produce more localized and complex boundaries.

Hyperparameter tuning is performed using grid search combined with cross-validation. This approach evaluates multiple combinations of parameter values across different data splits within the training subset, providing a stable estimate of relative model performance and reducing sensitivity to a particular partition of the data.

Given the class imbalance in the dataset, model comparison is based on PR-AUC (average precision), which focuses on the model's ability to correctly rank and identify churners. This metric is more informative than accuracy and complements ROC-AUC by emphasizing performance on the minority class.

Importantly, cross-validation is used as an internal model comparison tool rather than the final decision criterion. Because hyperparameter tuning is performed iteratively, relying solely on cross-validation results can lead to optimistic bias. To mitigate this, candidate models identified through cross-validation are evaluated on a separate validation set, which is used to guide final hyperparameter selection.

This two-stage approach separates baseline evaluation from model extension, ensuring that any observed performance improvements can be attributed to the introduction of non-linearity rather than differences in the optimization framework. The final model will subsequently be evaluated on a hold-out test set to obtain an unbiased estimate of performance.

In the following sections, we first define and tune the linear SVM, followed by the RBF SVM.

## Linear SVM

This section implements the linear Support Vector Machine (SVM) modeling workflow described above, translating the conceptual approach into a reusable and consistent pipeline.

Unlike logistic regression, a linear SVM does not model churn probabilities through an explicit parametric relationship between predictors and the log-odds of the outcome. Instead, it defines a linear decision boundary by maximizing the margin between classes, making classification depend on the relative positioning of observations in the feature space rather than on direct coefficient interpretation.

Although the resulting decision boundary is linear, the learning objective differs fundamentally from logistic regression. Rather than minimizing log-loss, the linear SVM optimizes a margin-based objective that emphasizes class separation and robustness. This makes it a useful benchmark for evaluating whether margin-based learning improves performance even when the decision surface remains linear.

The feature representation remains close to that used for earlier linear models in that categorical variables are encoded using one-hot encoding and numerical predictors are retained in continuous form. However, unlike the logistic regression specification, the linear SVM does not rely on manually defined interaction terms to begin with. This keeps the initial SVM formulation deliberately simple and allows the margin-based learning mechanism itself to be evaluated more directly.

Because SVMs are sensitive to the scale of the input variables, feature standardization is introduced as an essential preprocessing step. Without scaling, variables with larger numerical ranges can dominate the optimization process and distort the resulting decision boundary. Standardization ensures that all predictors contribute comparably to model fitting.

The training, validation, and test datasets are loaded and prepared using the shared data preparation pipeline. After feature construction, the predictor matrix is standardized using parameters estimated from the training subset and then applied consistently to the validation and test sets. This preserves the separation between model fitting and evaluation while avoiding data leakage.

Model training is performed using the `SVC` class from scikit-learn with a linear kernel. In this stage, model complexity is controlled primarily through the regularization parameter $C$, which determines the trade-off between maximizing the margin and minimizing classification errors on the training data.

To identify an appropriate model specification, candidate values of $C$ are explored using cross-validation within the training subset. This provides a systematic framework for comparing alternative regularization strengths while reducing sensitivity to any single partition of the data.

Because hyperparameter tuning is performed iteratively, cross-validation is used as an internal model comparison tool rather than the final selection criterion. Candidate models identified through cross-validation are subsequently evaluated on a separate validation set, which is used to select the final model configuration.

The implementation proceeds in the following steps:

1. Load training, validation, and test data
2. Apply the data preparation pipeline
3. Define target and predictor variables
4. Construct the linear SVM feature matrix
5. Standardize the predictor space using the training subset
6. Define the linear SVM model
7. Perform hyperparameter exploration using cross-validation on the training subset
8. Evaluate candidate models on the validation set and select the final configuration
9. Retrain the selected model on the combined training data (optional)
10. Evaluate final performance on the test set


Loading data:

In [1]:
import polars as pl
from sklearn.model_selection import train_test_split

# Load datasets
train_df = pl.read_csv("./data/processed/06_dpp_train_df.csv")
test_df = pl.read_csv("./data/processed/06_dpp_test_df.csv")

selected_cols = [
    'SeniorCitizenRelevel',
    'Partner',
    'Dependents',
    'tenure',
    'Contract',
    'PaperlessBilling',
    'MonthlyCharges',
    'Churn',
    'AdditionalInternetServicesCount',
    'StreamingServicesCount',
    'PaymentMethod_bin_3'
]

train_df = train_df.select(selected_cols)
test_df = test_df.select(selected_cols)

# Convert to pandas for sklearn
train_pd = train_df.to_pandas()

# Stratified split: 25% of current train -> validation
train_sub_pd, val_pd = train_test_split(
    train_pd,
    test_size=0.25,
    stratify=train_pd["Churn"],
    random_state=42
)

# Back to polars
train_sub_df = pl.from_pandas(train_sub_pd)
val_df = pl.from_pandas(val_pd)

# Shapes
print("Train subset shape:", train_sub_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Class proportions helper
def class_proportions(df, name):
    print(f"\n{name} class proportions:")
    print(
        df.group_by("Churn")
        .len()
        .with_columns(
            (pl.col("len") / pl.col("len").sum()).alias("proportion")
        )
        .sort("Churn")
    )

class_proportions(train_sub_df, "Train subset")
class_proportions(val_df, "Validation")
class_proportions(test_df, "Test")

Train subset shape: (4225, 11)
Validation shape: (1409, 11)
Test shape: (1409, 11)

Train subset class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 3104 ┆ 0.734675   │
│ Yes   ┆ 1121 ┆ 0.265325   │
└───────┴──────┴────────────┘

Validation class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘

Test class proportions:
shape: (2, 3)
┌───────┬──────┬────────────┐
│ Churn ┆ len  ┆ proportion │
│ ---   ┆ ---  ┆ ---        │
│ str   ┆ u32  ┆ f64        │
╞═══════╪══════╪════════════╡
│ No    ┆ 1035 ┆ 0.734564   │
│ Yes   ┆ 374  ┆ 0.265436   │
└───────┴──────┴────────────┘


In [2]:
categorical_cols = [
    "SeniorCitizenRelevel", "Partner", "Dependents", "Contract", "PaperlessBilling", "PaymentMethod_bin_3"
]

numerical_cols = [
    "tenure", "MonthlyCharges", "AdditionalInternetServicesCount", "StreamingServicesCount"
]

target_col = "Churn"

# Define the order of categories for categorical variables, first will be dropped when one hot encoding
categorical_orders = {
    "SeniorCitizenRelevel": ["No", "Yes"],
    "Partner": ["No", "Yes"],
    "Dependents": ["No", "Yes"],
    "PaperlessBilling": ["No", "Yes"],
    "Contract": ["Month-to-month", "One year", "Two year"]
}

# check if we are not forgetting any column
set(categorical_cols + numerical_cols + [target_col]) == set(train_df.columns)

True

In [3]:
from src.utils.data_preparation_models import prepare_linear_features

train_X, train_y = prepare_linear_features(
    df=train_sub_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    add_intercept = False,
    drop_first= False
)

validation_X, validation_y = prepare_linear_features(
    df=val_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    add_intercept = False,
    drop_first= False,
    reference_columns= train_X.columns.tolist()
)

### Feature Scaling

At this stage, the training and validation datasets have been constructed using the shared data preparation pipeline, resulting in aligned feature matrices for both subsets.

Before fitting the Support Vector Machine model, an additional preprocessing step is required. Unlike tree-based methods, SVMs are sensitive to the scale of input features, as the optimization of the decision boundary depends on distances in the feature space.

To ensure that all variables contribute proportionally to the model, features are standardized using z-score scaling. Each feature is transformed by subtracting its mean and dividing by its standard deviation, resulting in variables with approximately zero mean and unit variance.

The scaling parameters are estimated using the training data and then applied consistently to the validation and test sets. This approach preserves consistency across datasets while preventing data leakage, as information from the validation or test sets is not used during the fitting of the scaler.

With scaling in place, the feature matrices are ready for model training.

In [4]:
from src.utils.data_preparation_models import fit_scaler, transform_with_scaler

scaler = fit_scaler(train_X)

scaled_train_X = transform_with_scaler(train_X, scaler)
scaled_validation_X = transform_with_scaler(validation_X, scaler)

Note: One-hot encoded categorical variables are also included in the standardization process. While this transforms their original 0/1 representation, it ensures that all features—both binary and continuous—contribute on a comparable scale. This is particularly important for SVMs, where model behavior depends on distances and relative feature magnitudes rather than on the original encoding of individual variables.

Note: The target variable is encoded as 0 (no churn) and 1 (churn), consistent with previous models. While the theoretical formulation of SVM uses labels in {-1, +1}, scikit-learn handles this internally, allowing us to retain the original encoding without affecting model performance.

### Initial Grid Definition and Cross-Validation Linear

The linear Support Vector Machine model is initially explored using cross-validated grid search to identify promising values of the regularization parameter. This approach evaluates multiple configurations across different data splits within the training subset, providing a stable estimate of relative model performance.

Model comparison during this stage is based on PR-AUC (average precision), which is well-suited for imbalanced classification problems and emphasizes the model’s ability to correctly rank and identify churners.

For the linear SVM, the main hyperparameter of interest is $C$, which controls the trade-off between maximizing the margin and minimizing classification errors. Smaller values of $C$ impose stronger regularization and encourage a wider margin, while larger values allow the model to fit the training data more closely. Exploring a range of $C$ values therefore allows us to assess how sensitive model performance is to the strength of regularization.

Because the current stage focuses on the linear specification, the tuning process is intentionally restricted to this single parameter. This keeps the initial model exploration controlled and interpretable, allowing the performance of a margin-based linear classifier to be assessed before introducing additional complexity through a non-linear kernel.

At this stage, cross-validation is used as an internal model comparison tool rather than the final selection criterion. Candidate configurations identified through cross-validation will be further evaluated on a separate validation set to guide final model selection.

The tuning process therefore aims to identify values of $C$ that provide strong predictive performance while remaining stable across validation folds.

In [5]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "class_weight": [None, "balanced"]
}

The regularization parameter $C$ is explored over a logarithmic scale to capture a broad range of model behaviors, from strongly regularized models with wide margins to more flexible models that fit the training data more closely. This approach allows us to assess the sensitivity of the model to regularization strength without introducing unnecessary grid complexity.

In [6]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

svc = SVC(
    kernel="linear",
    probability=True,
    random_state=42
)

grid = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid.fit(scaled_train_X, train_y)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(kernel='l...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also 

In [7]:
import pandas as pd

cv_results = pd.DataFrame(grid.cv_results_)

cv_summary = (
    cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "params"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

cv_summary

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.650292,0.045110,0.649647,0.012756,"{'C': 1, 'class_weight': 'balanced'}"
1,2,0.649402,0.045292,0.648458,0.012163,"{'C': 10, 'class_weight': 'balanced'}"
2,3,0.649342,0.045172,0.649567,0.012781,"{'C': 0.1, 'class_weight': 'balanced'}"
3,4,0.648820,0.045802,0.648265,0.012200,"{'C': 100, 'class_weight': 'balanced'}"
4,5,0.648629,0.044847,0.648354,0.012376,"{'C': 0.01, 'class_weight': 'balanced'}"
5,6,0.646742,0.048289,0.648803,0.012233,"{'C': 10, 'class_weight': None}"
6,7,0.646739,0.047908,0.648536,0.011957,"{'C': 0.1, 'class_weight': None}"
7,8,0.646651,0.048265,0.648828,0.012214,"{'C': 1, 'class_weight': None}"
8,9,0.646567,0.048188,0.648815,0.012163,"{'C': 100, 'class_weight': None}"
9,10,0.643425,0.044694,0.645472,0.011851,"{'C': 0.01, 'class_weight': None}"


Based on cross-validation results, only the top-performing configurations are carried forward for validation. The differences in performance across configurations are relatively small, and lower-ranked models are consistently dominated by stronger alternatives.

Notably, all top-performing configurations include class weighting, while unweighted models consistently underperform. This confirms that accounting for class imbalance is essential for effective margin-based classification in this setting.

Focusing on a small subset of top candidates allows for a more targeted evaluation on the validation set while preserving interpretability. The selected configurations capture variation in regularization strength within the region where model performance is stable.

In [18]:
top_candidates = cv_summary.head(3).copy()

top_candidates

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.650292,0.045110,0.649647,0.012756,"{'C': 1, 'class_weight': 'balanced'}"
1,2,0.649402,0.045292,0.648458,0.012163,"{'C': 10, 'class_weight': 'balanced'}"
2,3,0.649342,0.045172,0.649567,0.012781,"{'C': 0.1, 'class_weight': 'balanced'}"


In [19]:
from sklearn.svm import SVC
from src.utils.classification_metrics import compute_classification_metrics

import pandas as pd


def evaluate_svm_candidates_on_validation(
    top_candidates,
    train_X,
    train_y,
    validation_X,
    validation_y,
    kernel,
    random_state=42,
    sort_by="pr_auc"
):
    """
    Fit SVM candidate models on training data and evaluate them on the validation set.

    Parameters
    ----------
    top_candidates : pd.DataFrame
        DataFrame containing candidate model results from cross-validation.
        Must include at least:
        - 'params'
        - 'rank_test_score'
        - 'mean_test_score'
        - 'std_test_score'

    train_X : pd.DataFrame
        Scaled training feature matrix.

    train_y : pd.Series or np.ndarray
        Training target values.

    validation_X : pd.DataFrame
        Scaled validation feature matrix.

    validation_y : pd.Series or np.ndarray
        Validation target values.

    kernel : str
        SVM kernel to use when fitting candidate models.

    random_state : int, default=42
        Random seed used for reproducibility.

    sort_by : str, default='pr_auc'
        Metric used to sort validation results.

    Returns
    -------
    pd.DataFrame
        Validation results for all candidate models, sorted by the chosen metric.
    """

    candidate_results = []

    for _, row in top_candidates.iterrows():
        params = row["params"]

        model = SVC(
            kernel=kernel,
            **params,
            random_state=random_state,
            probability=True,
        )

        model.fit(train_X, train_y)

        val_prob = model.predict_proba(validation_X)[:, 1]
        val_pred = model.predict(validation_X)

        metrics = compute_classification_metrics(
            y_true=validation_y,
            y_pred=val_pred,
            y_prob=val_prob
        )

        candidate_results.append({
            "cv_rank": row["rank_test_score"],
            "cv_pr_auc": row["mean_test_score"],
            "cv_std": row["std_test_score"],
            "params": params,
            **metrics
        })

    results_df = (
        pd.DataFrame(candidate_results)
        .sort_values(by=sort_by, ascending=False)
        .reset_index(drop=True)
    )

    return results_df

In [20]:
candidate_results_df = evaluate_svm_candidates_on_validation(
    top_candidates=top_candidates,
    train_X=scaled_train_X,
    train_y=train_y,
    validation_X=scaled_validation_X,
    validation_y=validation_y,
    kernel="linear"
)

candidate_results_df

,cv_rank,cv_pr_auc,cv_std,params,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,3,0.649342,0.045172,"{'C': 0.1, 'class_weight': 'balanced'}",0.693400,0.456716,0.818182,0.586207,0.809981,0.596337,306,671,364,68
1,1,0.650292,0.045110,"{'C': 1, 'class_weight': 'balanced'}",0.710433,0.472756,0.788770,0.591182,0.810658,0.596034,295,706,329,79
2,2,0.649402,0.045292,"{'C': 10, 'class_weight': 'balanced'}",0.710433,0.472756,0.788770,0.591182,0.810645,0.595996,295,706,329,79


The validation results show that performance is effectively unchanged across the tested regularization strengths ($C=0.1$, $C=1$, and $C=10$). Cross-validated PR AUC is highest at $C=1$, while validation PR AUC differs only marginally across all three settings, indicating a broad performance plateau rather than a sharp optimum.

Given this stability, $C=1$ is selected as the final linear SVM configuration. It provides the best cross-validated performance and achieves validation results that are practically indistinguishable from the alternatives, while avoiding the weaker regularization of $C=10$.

No further refinement of $C$ is performed, since both cross-validation and validation suggest that additional tuning would be unlikely to produce meaningful gains.

Because the refinement step did not reveal any meaningful performance differences within the explored region, additional validation is not required. The previously selected configuration ($C=1$, balanced class weighting) remains the final linear SVM candidate.

In [21]:
best_linear_svm_params = {'C': 1, 'class_weight': 'balanced'}

At this stage, the optimal configuration for the linear SVM has been identified based on cross-validation and validation performance. A subsequent refinement step confirmed that model performance is stable across a range of regularization strengths, indicating the presence of a broad performance plateau rather than a sharply defined optimum. The final configuration is therefore selected from within this region, favoring a more regularized and parsimonious model.

However, model evaluation is not yet finalized, as additional modeling approaches will be considered.

To maintain a consistent and unbiased comparison framework, final model training and test set evaluation are deferred until all candidate models have been developed and tuned. The next step therefore introduces a non-linear SVM using the RBF kernel, allowing us to assess whether controlled non-linearity provides meaningful improvements in predictive performance beyond the linear baseline.

## RBF kernel SVM

This section extends the SVM workflow by introducing a non-linear specification using the radial basis function (RBF) kernel.

The overall implementation remains consistent with the linear SVM setup described in the previous section. The same data preparation pipeline is retained, including one-hot encoding of categorical variables, continuous representation of numerical features, and feature standardization based on the training subset. This ensures that any performance differences can be attributed to the model specification itself rather than to changes in preprocessing or feature construction.

Where the RBF SVM differs is in the form of the decision boundary. Rather than restricting separation to a linear hyperplane in the original feature space, the RBF kernel allows the model to learn a flexible, non-linear boundary by implicitly mapping the data into a higher-dimensional representation. This enables the model to capture more complex relationships without requiring manually specified interaction terms or non-linear transformations.

As with the linear SVM, model training is performed using the `SVC` class from scikit-learn. However, in the RBF setting, model complexity is governed by two key hyperparameters. The regularization parameter $C$ controls the trade-off between margin width and classification error, while the parameter $\gamma$ determines how localized the influence of individual observations is in shaping the decision boundary. Together, these parameters determine the flexibility of the model and its tendency to underfit or overfit.

To preserve a structured modeling strategy, the RBF SVM is introduced only after establishing the linear SVM benchmark. This makes it possible to assess whether the additional flexibility provided by the kernel leads to meaningful improvements in predictive performance beyond what can be achieved through a linear margin-based classifier.

Hyperparameter tuning is again performed using cross-validation within the training subset, followed by validation-based model selection. As in the linear case, cross-validation serves as an internal comparison tool, while the separate validation set is used to guide final selection and reduce the risk of optimistic bias.

The implementation proceeds in the following steps:

1. Reuse the prepared and standardized feature matrices from the linear SVM workflow
2. Define the RBF SVM model
3. Perform hyperparameter exploration over $C$ and $\gamma$ using cross-validation on the training subset
4. Evaluate top candidate models on the validation set and select the final configuration
5. Defer final retraining and test set evaluation until all SVM variants have been compared

The goal of this section is therefore not to redefine the preprocessing pipeline, but to isolate the effect of controlled non-linearity within the same modeling framework.

### Initial Grid Definition and Cross-Validation RBF

The RBF Support Vector Machine model is explored using cross-validated grid search to identify suitable combinations of regularization strength and kernel flexibility. As in the linear case, this approach evaluates multiple configurations across different data splits within the training subset, providing a stable estimate of relative model performance.

Model comparison during this stage is based on PR-AUC (average precision), which is well-suited for imbalanced classification problems and emphasizes the model’s ability to correctly rank and identify churners.

In contrast to the linear SVM, the RBF specification introduces an additional hyperparameter. While the regularization parameter $C$ retains the same interpretation as before, the parameter $\gamma$ controls the influence of individual observations on the decision boundary. Smaller values of $\gamma$ produce smoother, more global decision surfaces, while larger values allow the model to fit more localized patterns in the data.

The hyperparameter grid is therefore constructed over combinations of $C$ and $\gamma$, allowing the tuning process to jointly explore the trade-off between margin-based regularization and the complexity of the non-linear decision boundary. This expands the search space relative to the linear model while remaining focused on the key drivers of model behavior.

Because the RBF SVM introduces substantially greater flexibility, care is taken to avoid unnecessarily large grids that would increase computational cost without providing additional insight. The grid is designed to cover a range of values on a logarithmic scale, capturing both strongly regularized, smooth models and more flexible, localized solutions.

As in previous stages, cross-validation is used as an internal model comparison tool rather than the final selection criterion. Candidate configurations identified through cross-validation will be further evaluated on a separate validation set to guide final model selection.

The tuning process therefore aims to identify combinations of $C$ and $\gamma$ that provide strong predictive performance while remaining stable across validation folds.

In [22]:
rbf_param_grid = {
    "C": [0.1, 1, 10, 50],
    "gamma": [0.001, 0.01, 0.1, 1],
    "class_weight": ["balanced"]  # retained based on linear SVM results
}

Based on the results from the linear SVM, class weighting is retained in the RBF specification. In the linear setting, unweighted models consistently underperformed and, under stronger regularization, failed to identify churners altogether. While this behavior has not been re-evaluated explicitly for the RBF kernel, it provides strong evidence that accounting for class imbalance is important for margin-based models in this problem.

The hyperparameter grid is therefore restricted to `class_weight="balanced"`, allowing the search to focus on the interaction between regularization and non-linear flexibility.

While the linear model indicated that moderate-to-high values of $C$ perform well, the introduction of the RBF kernel changes the relationship between regularization and model complexity. As a result, the grid reintroduces a broader range of $C$ values and jointly explores the kernel parameter $\gamma$ on a logarithmic scale.

This ensures that the tuning process captures both smooth and highly flexible decision boundaries without being overly constrained by assumptions derived from the linear setting.

In [13]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

rbf_svc = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

rbf_grid = GridSearchCV(
    estimator=rbf_svc,
    param_grid=rbf_param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

rbf_grid.fit(scaled_train_X, train_y)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(probabili...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...], 'class_weight': ['balanced'], 'gamma': [0.001, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 

In [14]:
import pandas as pd

rbf_cv_results = pd.DataFrame(rbf_grid.cv_results_)

rbf_cv_summary = (
    rbf_cv_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "params"
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

rbf_cv_summary

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.650956,0.046455,0.653153,0.012751,"{'C': 10, 'class_weight': 'balanced', 'gamma':..."
1,2,0.648805,0.043465,0.651041,0.010986,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'..."
2,3,0.647808,0.043711,0.647620,0.011371,"{'C': 1, 'class_weight': 'balanced', 'gamma': ..."
3,4,0.647127,0.045848,0.653390,0.013545,"{'C': 50, 'class_weight': 'balanced', 'gamma':..."
4,5,0.643084,0.044017,0.652290,0.014374,"{'C': 1, 'class_weight': 'balanced', 'gamma': ..."
5,6,0.623586,0.037295,0.622909,0.009496,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'..."
6,7,0.620240,0.033358,0.646677,0.012899,"{'C': 10, 'class_weight': 'balanced', 'gamma':..."
7,8,0.606364,0.034012,0.646745,0.012139,"{'C': 50, 'class_weight': 'balanced', 'gamma':..."
8,9,0.604799,0.033073,0.627761,0.012723,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'..."
9,10,0.595707,0.036509,0.664609,0.007461,"{'C': 1, 'class_weight': 'balanced', 'gamma': ..."


The results indicate that the RBF SVM achieves only marginal improvements over the linear specification. The best-performing configurations are associated with small values of $\gamma$, corresponding to smoother decision boundaries.

As $\gamma$ increases, model performance deteriorates rapidly, with substantial gaps between training and cross-validation scores, indicating overfitting. This suggests that the additional flexibility introduced by the RBF kernel does not provide meaningful benefits for this problem.

Overall, these findings indicate that the underlying decision boundary is well-approximated by a linear model, and that further increases in model complexity yield diminishing returns.

In [15]:
rbf_top_candidates = rbf_cv_summary.head(3).copy()

rbf_top_candidates

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
0,1,0.650956,0.046455,0.653153,0.012751,"{'C': 10, 'class_weight': 'balanced', 'gamma':..."
1,2,0.648805,0.043465,0.651041,0.010986,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'..."
2,3,0.647808,0.043711,0.647620,0.011371,"{'C': 1, 'class_weight': 'balanced', 'gamma': ..."


In [16]:
rbf_candidate_results_df = evaluate_svm_candidates_on_validation(
    top_candidates=rbf_top_candidates,
    train_X=scaled_train_X,
    train_y=train_y,
    validation_X=scaled_validation_X,
    validation_y=validation_y,
    kernel="rbf"
)

rbf_candidate_results_df

,cv_rank,cv_pr_auc,cv_std,params,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,3,0.647808,0.043711,"{'C': 1, 'class_weight': 'balanced', 'gamma': ...",0.641590,0.415045,0.855615,0.558952,0.806904,0.592611,320,584,451,54
1,1,0.650956,0.046455,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.671398,0.437058,0.826203,0.571693,0.808015,0.592464,309,637,398,65
2,2,0.648805,0.043465,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'...",0.644429,0.416337,0.844920,0.557811,0.804416,0.585410,316,592,443,58


The model with index 1 outperforms others in the precission and F1 measure while drop in pr_auc is only marginal.

In [23]:
rbf_selected_candidate = rbf_candidate_results_df.loc[1]
best_rbf_svm_params = rbf_selected_candidate["params"]
best_rbf_svm_params

{'C': 10, 'class_weight': 'balanced', 'gamma': 0.001}

The validation results identify the configuration with $C=10$ and $\gamma=0.001$ as the best-performing RBF SVM model.

Notably, the optimal value of $\gamma$ lies at the lower end of the explored range, indicating that the model benefits from smooth, globally consistent decision boundaries rather than highly localized flexibility. This suggests that while some non-linear structure is present in the data, it is relatively mild.

Further refinement of the hyperparameter grid is not pursued, as performance differences within this region are minimal, indicating a stable optimum. Additional tuning would therefore be unlikely to yield meaningful improvements.

### Training the best models on whole train dataset

In [24]:
import pandas as pd
from sklearn.svm import SVC

train_val_X = pd.concat([train_X, validation_X], axis=0)
train_val_y = pd.concat([train_y, validation_y], axis=0)

final_scaler = fit_scaler(train_val_X)

scaled_train_val_X = transform_with_scaler(train_val_X, final_scaler)

best_rbf_svc = SVC(
    **best_rbf_svm_params,
    kernel="rbf",
    probability=True,
    random_state=42
)

best_linear_svc = SVC(
    **best_linear_svm_params,
    kernel="linear",
    probability=True,
    random_state=42
)

best_rbf_svc.fit(scaled_train_val_X, train_val_y)
best_linear_svc.fit(scaled_train_val_X, train_val_y)


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


## Feature Importance

### Linear Kernel

For the linear SVM, feature importance is derived directly from the model coefficients. Because the input features are standardized, the magnitude of each coefficient reflects the relative importance of the corresponding variable in determining the decision boundary.

In [25]:
coef = best_linear_svc.coef_[0]

feature_importance = pd.DataFrame({
    "feature": train_val_X.columns,
    "coefficient": coef,
    "abs_coefficient": abs(coef)
}).sort_values("abs_coefficient", ascending=False)

feature_importance

,feature,coefficient,abs_coefficient
10,Contract_Month-to-month,0.579892,0.579892
12,Contract_Two year,-0.354643,0.354643
11,Contract_One year,-0.337787,0.337787
0,tenure,-0.000879,0.000879
1,MonthlyCharges,0.000760,0.000760
2,AdditionalInternetServicesCount,-0.000488,0.000488
3,StreamingServicesCount,0.000156,0.000156
16,PaymentMethod_bin_3_Electronic check,0.000151,0.000151
17,PaymentMethod_bin_3_Mailed check,-0.000127,0.000127
14,PaperlessBilling_Yes,0.000103,0.000103


The results highlight several key drivers of churn:

- **Tenure** is the most influential factor, with longer-tenured customers significantly less likely to churn.
- **Monthly charges** have a strong positive association with churn, indicating that higher-priced plans increase the likelihood of customer attrition.
- **Contract type** plays a critical role, with month-to-month contracts strongly associated with higher churn, while longer-term contracts reduce churn risk.

Additional factors such as the number of subscribed services, payment method, and billing preferences also contribute to churn behavior, though with smaller effects.

Overall, the model identifies a consistent and interpretable set of drivers, reflecting both economic factors (price sensitivity) and behavioral patterns (customer commitment and service engagement).

### RBF Kernel

Unlike the linear SVM, the RBF SVM does not yield directly interpretable coefficients, as the model relies on a non-linear transformation of the feature space. Consequently, feature importance cannot be obtained from model parameters.

Instead, permutation-based feature importance is used to evaluate the contribution of each feature. This method measures the impact on model performance when individual features are randomly shuffled, providing a model-agnostic assessment of feature relevance.

In [26]:
from sklearn.inspection import permutation_importance
import pandas as pd

rbf_permutation = permutation_importance(
    estimator=best_rbf_svc,
    X=scaled_train_val_X,
    y=train_val_y,
    scoring="average_precision",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

rbf_feature_importance = (
    pd.DataFrame({
        "feature": scaled_train_val_X.columns,
        "importance_mean": rbf_permutation.importances_mean,
        "importance_std": rbf_permutation.importances_std
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

rbf_feature_importance

,feature,importance_mean,importance_std
0,tenure,0.102606,0.005387
1,MonthlyCharges,0.097412,0.009633
2,Contract_Month-to-month,0.089741,0.009648
3,Contract_Two year,0.047280,0.005904
4,Contract_One year,0.036399,0.007145
5,AdditionalInternetServicesCount,0.034427,0.006537
6,PaperlessBilling_No,0.004031,0.002346
7,PaperlessBilling_Yes,0.004031,0.002346
8,PaymentMethod_bin_3_Electronic check,0.003701,0.002339
9,Dependents_No,0.001257,0.000615


Permutation-based feature importance reveals a structure that closely mirrors the results obtained from the linear SVM. The most influential variables—tenure, monthly charges, and contract type—remain dominant, followed by the number of subscribed services.

This consistency indicates that the RBF SVM does not rely on fundamentally different features, but rather refines the decision boundary within the same underlying feature space. While the non-linear model achieves modest improvements in predictive performance, particularly in recall, it does not introduce new drivers of churn.

Overall, these results suggest that the primary relationships governing churn are largely captured by the existing feature set and are well approximated by a linear structure, with only limited benefit from additional non-linearity.

### Model Serialization

The final linear and RBF SVM models are serialized together with the fitted scaler and feature column structure. This ensures that future data can be transformed consistently before prediction and that the models can be deployed without additional preprocessing steps.

Each model is stored as a self-contained artifact, including the trained estimator, preprocessing components, and metadata describing the model configuration.

In [27]:
import pickle

reference_columns = train_val_X.columns.tolist()

# Linear SVM artifact
linear_artifact = {
    "model_type": "linear_svm",
    # model
    "model": best_linear_svc,
    # preprocessing config
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "target_col": target_col,
    "reference_columns": reference_columns,
    # scaler
    "scaler": final_scaler,
    # model params
    "params": best_linear_svm_params
}

# RBF SVM artifact
rbf_artifact = {
    "model_type": "rbf_svm",
    # model
    "model": best_rbf_svc,
    # preprocessing config
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "target_col": target_col,
    "reference_columns": reference_columns,
    # scaler
    "scaler": final_scaler,
    # model params
    "params": best_rbf_svm_params 
}

with open("./models/linear_svm_final.pkl", "wb") as f:
    pickle.dump(linear_artifact, f)

with open("./models/rbf_svm_final.pkl", "wb") as f:
    pickle.dump(rbf_artifact, f)

## Test Set Evaluation

The final linear and RBF SVM models are evaluated on the hold-out test set, which has remained untouched throughout model development, tuning, and interpretation. This provides an unbiased estimate of out-of-sample performance and enables a fair comparison between the linear and non-linear specifications.

The test features are transformed using the scaler fitted on the combined training and validation data, ensuring consistency with the final model training pipeline. The resulting metrics summarize the extent to which the selected models generalize to unseen customers.

In [28]:
test_X, test_y = prepare_linear_features(
    df=test_df,
    categorical_cols=categorical_cols,
    numerical_cols=numerical_cols,
    categorical_orders=categorical_orders,
    add_intercept = False,
    drop_first= False,
    reference_columns= reference_columns
)

scaled_test_X = transform_with_scaler(test_X, final_scaler)

# Linear SVM
linear_test_prob = best_linear_svc.predict_proba(scaled_test_X)[:, 1]
linear_test_pred = best_linear_svc.predict(scaled_test_X)

linear_test_metrics = compute_classification_metrics(
    y_true=test_y,
    y_pred=linear_test_pred,
    y_prob=linear_test_prob
)

# RBF SVM
rbf_test_prob = best_rbf_svc.predict_proba(scaled_test_X)[:, 1]
rbf_test_pred = best_rbf_svc.predict(scaled_test_X)

rbf_test_metrics = compute_classification_metrics(
    y_true=test_y,
    y_pred=rbf_test_pred,
    y_prob=rbf_test_prob
)

# comparison table
test_results = pd.DataFrame([
    {"model": "Linear SVM", **linear_test_metrics},
    {"model": "RBF SVM", **rbf_test_metrics}
]).sort_values("pr_auc", ascending=False).reset_index(drop=True)

test_results

,model,accuracy,precision,recall,f1,auc,pr_auc,tp,tn,fp,fn
0,RBF SVM,0.660043,0.430464,0.868984,0.575731,0.832818,0.658481,325,605,430,49
1,Linear SVM,0.659333,0.430079,0.871658,0.575972,0.830784,0.652776,326,603,432,48


Both the linear and RBF SVM models deliver very similar performance on the hold-out test set, with only marginal differences across all evaluation metrics. Their ROC-AUC and PR-AUC values are close, indicating that both models have comparable ability to rank customers by churn risk.

The RBF SVM achieves slightly higher ROC-AUC and PR-AUC, suggesting a small advantage in overall ranking performance. In contrast, the linear SVM yields marginally higher recall and F1 score, indicating a slightly better balance between identifying churners and maintaining classification quality at the selected threshold.

Overall, the differences between the two specifications are negligible in practical terms. These results suggest that introducing a non-linear decision boundary does not materially improve predictive performance over the linear alternative for this churn prediction task.

Given the absence of a clear test-set winner, both SVM variants are retained as candidate models for comparison with the other modeling approaches developed in this project. Final model selection will therefore be based on the unified comparison across all candidate models.

## Executive Summary

This project developed and evaluated predictive models to identify customers at risk of churn, with the goal of producing reliable probability estimates and actionable customer rankings.

The modeling workflow followed a structured approach, progressing from interpretable linear models to more flexible non-linear methods. Each model was evaluated using a consistent framework based on cross-validation, validation, and a final hold-out test set to ensure unbiased performance assessment.

Across all experiments, a small set of features consistently emerged as the primary drivers of churn. In particular, customer tenure, monthly charges, and contract type were found to have the strongest influence on churn behavior, reflecting a combination of customer commitment and price sensitivity. Additional factors, such as the number of subscribed services and payment method, contributed more modestly.

Support Vector Machines were explored in both linear and non-linear (RBF) forms. The linear model provided strong and stable performance, while the RBF variant introduced a slight improvement in recall and PR-AUC, indicating a modest benefit from controlled non-linearity. However, these gains were relatively small, and both models demonstrated similar overall predictive capability on the test set.

Importantly, the analysis showed that the underlying structure of the problem is largely well captured by a linear decision boundary, with non-linear extensions providing only incremental improvements. Feature importance analysis further confirmed that both linear and non-linear models rely on the same core set of variables.

At this stage, multiple candidate models have been developed and evaluated, including both linear and non-linear specifications. Final model selection is deferred to a unified comparison across all approaches, ensuring that the chosen solution reflects a consistent and unbiased evaluation.

Overall, the project demonstrates that a well-structured modeling pipeline, combined with careful validation and interpretability analysis, can produce robust and actionable insights into customer churn behavior.